In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec # Needed for better spacing control
import numpy as np
from qutip import *

In [ ]:
# ==========================================
# 1. TEXTBOOK FONT CONFIGURATION (The Fix)
# ==========================================
# We use 'stix' for math, which mimics the 'txfonts' package in your LaTeX.
# We use 'Times New Roman' for regular text to match the body.
textbook_params = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",  # Crucial: Makes math look like LaTeX
    "axes.labelsize": 14,
    "font.size": 12,
    "legend.fontsize": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.titlesize": 16,
}
plt.rcParams.update(textbook_params)

In [ ]:
# Define Hilbert space size
N = 35 

def plot_wigner_publication_ready(psi_list, titles=None, filename="wigner_final.pdf"):
    xvec = np.linspace(-6, 6, 150)
    X, Y = np.meshgrid(xvec, xvec)
    
    # FIGURE SIZE: Increased width to 22 to give subplots more "breathing room"
    fig = plt.figure(figsize=(22, 7))
    
    # SPACING: Increased wspace to 0.4 to prevent Y-axis overlap
    # In 3D plots, the bounding boxes are large, so they need more space than 2D plots.
    gs = gridspec.GridSpec(1, 3, wspace=-0.1)
    
    # COLOR LIMITS: Forced to +/- 0.4. 
    # This ensures 0 is mathematically the "center" (pure white) in 'RdBu_r'
    clim = 0.4
    my_levels = np.linspace(-clim, clim, 100)

    for i, psi in enumerate(psi_list):
        ax = fig.add_subplot(gs[i], projection='3d')
        ax.set_facecolor('white')
        
        W = wigner(psi, xvec, xvec)
        
        # 1. THE FLOOR PROJECTION (The "Shadow")
        # 'offset' puts it at the very bottom of the Z-axis box
        ax.contourf(X, Y, W, zdir='z', offset=-clim, 
                    cmap='RdBu_r', 
                    levels=my_levels, 
                    vmin=-clim, vmax=clim, 
                    alpha=1.0) 

        # 2. THE 3D SURFACE
        ax.plot_surface(X, Y, W, cmap='RdBu_r', 
                        vmin=-clim, vmax=clim,
                        rstride=1, cstride=1,
                        linewidth=0, shade=True, antialiased=True)

        # 3. CLEANING & STYLING
        ax.set_zlim(-clim, clim)
        
        # REMOVE GRID: This removes the dotted/solid interior lines
        ax.grid(False)
        
        # TRANSPARENT PANES: Removes the grey shading on the walls
        ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        
        # LABELS with padding to avoid clashing with the plot
        ax.set_xlabel(r'$q$', labelpad=12, style='italic')
        ax.set_ylabel(r'$p$', labelpad=12, style='italic')

        # VIEW ANGLE: Slightly flatter angle often looks better for projections
        ax.view_init(elev=22, azim=-45)
        
        if titles:
            ax.set_title(titles[i], pad=0, fontsize=22)

    # Final margin adjustment to ensure labels aren't cut off at the edges
    plt.subplots_adjust(left=0.05, right=0.95, top=0.85, bottom=0.1)
    
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    print(f"Plot successfully saved to {filename}")
    plt.show()

# --- Define States ---
psi_fock1 = basis(N, 1)
psi_fock3 = basis(N, 3)
# Odd Cat State (normalized)
psi_cat = (coherent(N, 2.5) - coherent(N, -2.5)).unit()

states = [psi_fock1, psi_fock3, psi_cat]
labels = [r'Fock State $|1\rangle$', r'Fock State $|3\rangle$', r'Odd Cat State']

# Generate the plot
plot_wigner_publication_ready(states, labels)

In [ ]:
# Define Hilbert space size
N = 30 

def plot_wigner_pdf(psi_list, titles=None, filename="wigner_final.pdf"):
    xvec = np.linspace(-6, 6, 150) 
    X, Y = np.meshgrid(xvec, xvec)
    
    fig = plt.figure(figsize=(20, 7)) 
    fig.patch.set_facecolor('white')  
    
    gs = gridspec.GridSpec(1, 3, wspace=0.05, width_ratios=[1, 1, 1])
    
    # --- RESTORE ORIGINAL COLORS ---
    # QuTiP standard style is usually shaded with a Red-Blue map.
    # We use 'RdBu' (Red-Blue).
    # Matplotlib's 'RdBu' goes from Red (low) to Blue (high).
    # Wigner convention is usually Blue (neg) to Red (pos).
    # So we use 'RdBu_r' (reversed) so Blue is low (neg) and Red is high (pos).
    my_cmap = plt.get_cmap('RdBu_r') 
    
    for i, psi in enumerate(psi_list):
        ax = plt.subplot(gs[i], projection='3d')
        ax.set_facecolor('white') 
        
        W = wigner(psi, xvec, xvec)
        
        # PLOT SURFACE (Original 3D Look)
        # shade=True: Restores the 3D lighting/shadows (Glossy look)
        # linewidth=0.1: Adds a very faint grid mesh for texture (optional, looks nice in PDF)
        surf = ax.plot_surface(X, Y, W, 
                               cmap=my_cmap, 
                               vmin=-0.3, vmax=0.3, # Symmetric limits ensure 0 is White
                               rstride=1, cstride=1,
                               linewidth=0,        # Set to 0.1 if you want faint grid lines
                               shade=True,         # <--- KEY CHANGE: Restores 3D lighting
                               antialiased=True)

        # CLEAN BACKGROUND
        ax.grid(False)
        ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
        ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
        ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
        
        edge_color = (0.9, 0.9, 0.9, 1.0) 
        ax.xaxis.pane.set_edgecolor(edge_color)
        ax.yaxis.pane.set_edgecolor(edge_color)
        ax.zaxis.pane.set_edgecolor(edge_color)

        # LABELS
        ax.set_xlabel(r'$q$', labelpad=10, fontsize=18, style='italic')
        ax.set_ylabel(r'$p$', labelpad=10, fontsize=18, style='italic')

        ax.view_init(elev=25, azim=-45)
        
        if titles:
            ax.set_title(titles[i], y=1.1)

    plt.subplots_adjust(left=0.02, right=0.98, top=0.85, bottom=0.1)
    
    # SAVE AS PDF
    plt.savefig(filename, format='pdf', bbox_inches='tight', facecolor=fig.get_facecolor())
    print(f"Saved to {filename}")
    plt.show()

In [ ]:
# --- Define States ---
psi_fock1 = basis(N, 1)
psi_fock3 = basis(N, 3)
psi_cat = (coherent(N, 2.5) - coherent(N, -2.5)).unit()

states = [psi_fock1, psi_fock3, psi_cat]
labels = [r'Fock State $|1\rangle$', r'Fock State $|3\rangle$', r'Odd Cat State']

plot_wigner_pdf(states, titles=labels)

In [ ]:
# ==========================================
# 1. TEXTBOOK FONT CONFIGURATION
# ==========================================
textbook_params = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",  # Mimics LaTeX 'txfonts'
    "axes.labelsize": 14,
    "font.size": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.titlesize": 16,
}
plt.rcParams.update(textbook_params)

# Define Hilbert space size
N = 30 

# --- HELPER: Transparent Colormap ---
def get_transparent_bwr():
    n_colors = 256
    bwr = plt.get_cmap('bwr', n_colors)
    new_colors = bwr(np.linspace(0, 1, n_colors))
    alpha_curve = np.abs(np.linspace(-1, 1, n_colors))**0.5 
    new_colors[:, 3] = alpha_curve 
    return mcolors.ListedColormap(new_colors)

def plot_textbook_wigner_fixed(psi_list, titles=None, filename="wigner_textbook_style.png"):
    xvec = np.linspace(-6, 6, 150) 
    X, Y = np.meshgrid(xvec, xvec)
    
    # 2. SETUP FIGURE
    fig = plt.figure(figsize=(20, 7)) 
    fig.patch.set_facecolor('white')  
    
    # GridSpec for spacing control
    # wspace=0.3 gives breathing room between plots
    gs = gridspec.GridSpec(1, 3, wspace=0.3, width_ratios=[1, 1, 1])
    
    my_cmap = get_transparent_bwr()
    
    for i, psi in enumerate(psi_list):
        ax = plt.subplot(gs[i], projection='3d')
        ax.set_facecolor('white') 
        
        W = wigner(psi, xvec, xvec)
        
        # PLOT SURFACE
        surf = ax.plot_surface(X, Y, W, 
                               cmap=my_cmap, 
                               vmin=-0.3, vmax=0.3,
                               rstride=1, cstride=1,
                               linewidth=0, shade=False)

        # CLEAN WHITE BACKGROUND
        ax.grid(False)
        ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
        ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
        ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
        
        edge_color = (0.9, 0.9, 0.9, 1.0) 
        ax.xaxis.pane.set_edgecolor(edge_color)
        ax.yaxis.pane.set_edgecolor(edge_color)
        ax.zaxis.pane.set_edgecolor(edge_color)

        # LABELS
        # Using raw strings r''
        ax.set_xlabel(r'Re($\alpha$)', labelpad=10)
        ax.set_ylabel(r'Im($\alpha$)', labelpad=10)

        # View Angle
        ax.view_init(elev=25, azim=-45)
        
        # TITLES (Pushed up)
        if titles:
            ax.set_title(titles[i], y=1.1)

    # 3. MANUAL LAYOUT ADJUSTMENT (Fixes the UserWarning)
    # Instead of plt.tight_layout(), we manually trim the margins.
    # top=0.85 ensures titles don't get cut off.
    # bottom=0.1 ensures labels don't get cut off.
    plt.subplots_adjust(left=0.05, right=0.95, top=0.85, bottom=0.1)
    
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

# --- Define States ---
psi_fock1 = basis(N, 1)
psi_fock3 = basis(N, 3)
psi_cat = (coherent(N, 2.5) - coherent(N, -2.5)).unit()

states = [psi_fock1, psi_fock3, psi_cat]

# CORRECTION: Replaced \ket{1} with |1\rangle
labels = [r'Fock State $|1\rangle$', r'Fock State $|3\rangle$', r'Odd Cat State']

plot_textbook_wigner_fixed(states, titles=labels)

In [ ]:
# --- Define States ---
psi_fock1 = basis(N, 1)
psi_fock3 = basis(N, 3)
psi_cat = (coherent(N, 2.5) - coherent(N, -2.5)).unit()

states = [psi_fock1, psi_fock3, psi_cat]
labels = [r'Fock State $\ket{1}$', r'Fock State $\ket{3}$', r'Odd Cat State']

plot_textbook_wigner_final(states, titles=labels)

In [ ]:
def save_textbook_wigner(filename="wigner_plot.png"):
    fig = plt.figure(figsize=(18, 5)) # Slightly wider for textbook layout
    
    # Define the 3 states
    states = [
        basis(N, 1), # Fock |1>
        basis(N, 3), # Fock |3>
        (coherent(N, 2.5) - coherent(N, -2.5)).unit() # Odd Cat
    ]
    titles = [r'Fock State $|1\rangle$', r'Fock State $|3\rangle$', r'Odd Cat State']
    
    for i, psi in enumerate(states):
        ax = fig.add_subplot(1, 3, i+1, projection='3d')
        W = wigner(psi, xvec, xvec)
        
        # Plot Surface (Clean, no mesh, white zero)
        surf = ax.plot_surface(X, Y, W, cmap='seismic', vmin=-0.3, vmax=0.3,
                               rstride=1, cstride=1, alpha=0.9, linewidth=0, antialiased=False)

        # Clean Background (Textbook Style)
        ax.grid(False) # No grid lines
        ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('w'); ax.yaxis.pane.set_edgecolor('w'); ax.zaxis.pane.set_edgecolor('w')

        # Axes Labels
        ax.set_xlabel(r'Re($\alpha$)', fontsize=14, labelpad=5)
        ax.set_ylabel(r'Im($\alpha$)', fontsize=14, labelpad=5)
        # ax.set_zlabel(r'$W$', fontsize=14) # Z-label often overlaps, optional
        
        ax.view_init(elev=30, azim=-45)
        ax.set_title(titles[i], fontsize=16, y=1.0)
        
        # Remove tick labels if you want a cleaner look, or keep them:
        # ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])

    plt.tight_layout()
    
    # SAVE THE FIGURE
    # dpi=300 is standard for print textbooks. 
    # transparent=True makes the background mix well with the page.
    plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f"Saved to {filename}")

save_textbook_wigner("wigner_negativity_compare.png")